# 🎓 Student Math Score Prediction — Linear Regression Pipeline

![Python](https://img.shields.io/badge/Python-3.8+-blue?logo=python)
![scikit-learn](https://img.shields.io/badge/scikit--learn-1.0+-orange?logo=scikit-learn)
![Task](https://img.shields.io/badge/Task-Regression-purple)
![Dataset](https://img.shields.io/badge/Dataset-Student%20Performance-lightgrey)

---

## 📌 Project Overview

This project predicts a student's **math score** from demographic and academic preparation data using a full **scikit-learn Pipeline** with `ColumnTransformer`.

| Item | Detail |
|------|--------|
| **Dataset** | Students Performance in Exams |
| **Model** | Linear Regression |
| **Target** | `math score` (0–100) |
| **Key Skill** | sklearn Pipeline with mixed feature types |

## 📦 1. Import Libraries

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print('✅ Libraries loaded successfully!')

## 📂 2. Load & Explore Data

> 📥 Dataset: [Students Performance in Exams — Kaggle](https://www.kaggle.com/datasets/spscientist/students-performance-in-exams)
>
> Download `StudentsPerformance.csv` and place it in the same folder as this notebook.

In [ ]:
# Auto-detect file format
if os.path.exists('StudentsPerformance.csv'):
    data = pd.read_csv('StudentsPerformance.csv')
    print('✅ Loaded StudentsPerformance.csv')
elif os.path.exists('StudentsPerformance.xlsx'):
    data = pd.read_excel('StudentsPerformance.xlsx')
    print('✅ Loaded StudentsPerformance.xlsx')
else:
    raise FileNotFoundError('❌ Please place StudentsPerformance.csv or .xlsx in this folder!')

print(f'Shape: {data.shape}')
data.head(10)

In [ ]:
# Basic info
data.info()

In [ ]:
# Statistical summary
data.describe()

In [ ]:
# Math score distribution
plt.figure(figsize=(8, 4))
sns.histplot(data['math score'], kde=True, color='steelblue', bins=30)
plt.title('Math Score Distribution')
plt.xlabel('Math Score')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Math score by gender
plt.figure(figsize=(7, 4))
sns.boxplot(x='gender', y='math score', data=data, palette='Set2')
plt.title('Math Score by Gender')
plt.tight_layout()
plt.show()

In [ ]:
# Math score by test preparation course
plt.figure(figsize=(7, 4))
sns.boxplot(x='test preparation course', y='math score', data=data, palette='Set3')
plt.title('Math Score by Test Preparation Course')
plt.tight_layout()
plt.show()

In [ ]:
# Math score by parental education level
plt.figure(figsize=(10, 4))
order = ["some high school", "high school", "some college",
         "associate's degree", "bachelor's degree", "master's degree"]
sns.boxplot(x='parental level of education', y='math score',
            data=data, order=order, palette='Blues')
plt.title('Math Score by Parental Education Level')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

## ✂️ 3. Split Data — Train & Test

In [ ]:
target = 'math score'
X = data.drop(target, axis=1)
y = data[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train size: {X_train.shape[0]} samples')
print(f'Test size : {X_test.shape[0]} samples')

## 🔧 4. Build Pipeline

We handle **3 types of features**:

| Type | Features | Transformer |
|------|----------|-------------|
| Numerical | reading score, writing score | Imputer + StandardScaler |
| Ordinal | education level, gender, lunch, test prep | Imputer + OrdinalEncoder |
| Nominal | race/ethnicity | Imputer + OneHotEncoder |

In [ ]:
# Define feature groups
num_features = ['reading score', 'writing score']
nom_features = ['race/ethnicity']
ord_features = ['parental level of education', 'gender', 'lunch', 'test preparation course']

# Ordinal categories (order matters for education level)
education_values = [
    'some high school', 'high school', 'some college',
    "associate's degree", "bachelor's degree", "master's degree"
]
gender_values  = X_train['gender'].unique().tolist()
lunch_values   = X_train['lunch'].unique().tolist()
test_prep_vals = X_train['test preparation course'].unique().tolist()

# Numerical pipeline
num_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

# Ordinal pipeline
ord_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(
        categories=[education_values, gender_values, lunch_values, test_prep_vals]
    ))
])

# Nominal pipeline
nom_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(sparse_output=False, handle_unknown='ignore'))
])

# Combine all transformers
preprocessor = ColumnTransformer(transformers=[
    ('num', num_transformer, num_features),
    ('nom', nom_transformer, nom_features),
    ('ord', ord_transformer, ord_features),
])

# Full pipeline with model
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

print('✅ Pipeline built successfully!')
print(pipeline)

## 🏋️ 5. Train Model

In [ ]:
pipeline.fit(X_train, y_train)
print('✅ Model trained successfully!')

## 📊 6. Evaluate Model

In [ ]:
y_pred = pipeline.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)

print('='*35)
print(f'  MAE  : {mae:.4f}')
print(f'  MSE  : {mse:.4f}')
print(f'  R²   : {r2:.4f}')
print('='*35)

In [ ]:
# Actual vs Predicted scatter plot
plt.figure(figsize=(8, 5))
plt.scatter(y_test, y_pred, alpha=0.5, color='steelblue', edgecolors='k', linewidths=0.3)
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Math Score')
plt.ylabel('Predicted Math Score')
plt.title('Actual vs Predicted — Math Score')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Residuals plot
residuals = y_test - y_pred

plt.figure(figsize=(8, 4))
plt.scatter(y_pred, residuals, alpha=0.5, color='tomato', edgecolors='k', linewidths=0.3)
plt.axhline(0, color='black', linestyle='--', lw=1.5)
plt.xlabel('Predicted Math Score')
plt.ylabel('Residual')
plt.title('Residual Plot')
plt.tight_layout()
plt.show()

In [ ]:
# Sample predictions vs actual
comparison = pd.DataFrame({
    'Actual'   : y_test.values[:15],
    'Predicted': y_pred[:15].round(1),
    'Diff'     : (y_test.values[:15] - y_pred[:15]).round(1)
})
print('Sample Predictions vs Actual (first 15):')
print(comparison.to_string(index=False))

## 🏁 7. Summary

| Item | Detail |
|------|--------|
| **Model** | Linear Regression |
| **Pipeline** | ColumnTransformer + sklearn Pipeline |
| **Numerical Features** | reading score, writing score |
| **Ordinal Features** | education level, gender, lunch, test prep |
| **Nominal Features** | race/ethnicity |

### 🔑 Key Takeaways
- ✅ **Reading & Writing scores** are the strongest predictors of math score
- ✅ **Test preparation course** completion positively impacts scores
- ✅ **Higher parental education** correlates with better student performance
- ✅ sklearn `Pipeline` ensures **no data leakage** between train and test sets
- ✅ `ColumnTransformer` handles different feature types cleanly in one step